<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03b_feature_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03b_feature_application**

## **Introducción**

Esta notebook corresponde al stage_03 - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación e importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [3]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [4]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

### 0.3. Definición de rutas



In [5]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [81]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/02_processed/mnq_intraday.parquet"))
#IN_RAW_PARQUET = Path(os.environ.get("IN_RAW_PARQUET", "data/01_raw/mnq_intraday.parquet"))
#IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))



In [80]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT}



### 0.4. Códigos auxiliares para carga de datos y visualización


In [8]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [9]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

# **1. Carga de dataset**

In [10]:
mnq_intraday = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (894845, 8)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'regime_id']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2025-06-13 20:00:00+00:00


### Validación temporal de dataset `mnq_intraday`


In [11]:
import pandas as pd
import numpy as np

# ============================================================
# Validación temporal de mnq_intraday
# Ejecutar inmediatamente después de:
# mnq_intraday = load_mnq_parquet()
# ============================================================

def validate_mnq_intraday(df: pd.DataFrame, verbose: bool = True) -> dict:
    """
    Valida consistencia temporal básica para un dataset intradía.

    Chequeos:
    1) Índice datetime válido
    2) Orden cronológico global
    3) Duplicados de timestamp
    4) Consistencia de columna `date`
    5) Monotonía de `minute_of_day` dentro de cada día
    6) Duplicados de `minute_of_day` dentro de cada día
    7) Saltos temporales negativos o nulos
    8) Gaps intradía distintos de 1 minuto
    """

    result = {
        "ok": True,
        "checks": {},
        "summary": {},
        "artifacts": {}
    }

    df = df.copy()

    # ------------------------------------------------------------
    # 0) Verificaciones básicas de estructura
    # ------------------------------------------------------------
    required_cols = ["date", "minute_of_day", "open", "high", "low", "close", "volume"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser un pd.DatetimeIndex")

    if df.empty:
        raise ValueError("El DataFrame está vacío")

    # ------------------------------------------------------------
    # 1) Orden global del índice
    # ------------------------------------------------------------
    is_monotonic = df.index.is_monotonic_increasing
    has_unique_index = df.index.is_unique
    duplicated_index = df.index[df.index.duplicated()].unique()

    result["checks"]["index_is_monotonic_increasing"] = bool(is_monotonic)
    result["checks"]["index_is_unique"] = bool(has_unique_index)
    result["summary"]["n_duplicated_timestamps"] = int(len(duplicated_index))
    result["artifacts"]["duplicated_timestamps"] = duplicated_index

    # Si no está ordenado, mostramos evidencia pero no reordenamos silenciosamente
    if not is_monotonic:
        diffs_ns = pd.Series(df.index.view("i8")).diff()
        bad_order_pos = np.where(diffs_ns <= 0)[0]
        result["artifacts"]["bad_global_order_positions"] = bad_order_pos[:20]
        result["ok"] = False

    if not has_unique_index:
        result["ok"] = False

    # ------------------------------------------------------------
    # 2) Consistencia entre index.date y columna `date`
    # ------------------------------------------------------------
    # Normalizamos ambos a fecha sin hora
    index_dates = pd.Index(df.index.tz_localize(None).date if df.index.tz is not None else df.index.date)
    col_dates = pd.to_datetime(df["date"]).dt.date

    date_match = (index_dates == col_dates).all()
    result["checks"]["date_column_matches_index_date"] = bool(date_match)

    if not date_match:
        mismatch_mask = index_dates != col_dates
        mismatches = df.loc[mismatch_mask, ["date", "minute_of_day", "close"]].head(20)
        result["artifacts"]["date_mismatches_head"] = mismatches
        result["summary"]["n_date_mismatches"] = int(mismatch_mask.sum())
        result["ok"] = False
    else:
        result["summary"]["n_date_mismatches"] = 0

    # ------------------------------------------------------------
    # 3) Diferencias temporales globales
    # ------------------------------------------------------------
    # Trabajamos en segundos
    diffs_sec = pd.Series(df.index).diff().dt.total_seconds()

    n_non_positive_diffs = int((diffs_sec.iloc[1:] <= 0).sum())
    result["checks"]["all_global_time_diffs_positive"] = (n_non_positive_diffs == 0)
    result["summary"]["n_non_positive_global_diffs"] = n_non_positive_diffs

    if n_non_positive_diffs > 0:
        bad_diff_rows = df.iloc[np.where((diffs_sec <= 0).fillna(False))[0][:20]]
        result["artifacts"]["non_positive_global_diffs_head"] = bad_diff_rows
        result["ok"] = False

    # ------------------------------------------------------------
    # 4) Validación por día
    # ------------------------------------------------------------
    daily_stats = []
    bad_minute_order_days = []
    duplicate_minute_days = []
    intraday_gap_rows = []

    grouped = df.groupby("date", sort=False)

    for day, g in grouped:
        g = g.copy()

        # 4.1 orden del índice dentro del día
        idx_mono = g.index.is_monotonic_increasing

        # 4.2 minute_of_day creciente dentro del día
        mod_diff = g["minute_of_day"].diff()
        minute_order_ok = bool((mod_diff.iloc[1:] > 0).all())

        # 4.3 duplicados de minute_of_day dentro del día
        dup_mod = g["minute_of_day"].duplicated().sum()
        has_dup_mod = dup_mod > 0

        # 4.4 gaps intradía del índice
        idx_diff_sec = pd.Series(g.index).diff().dt.total_seconds()
        gap_mask = (~idx_diff_sec.isna()) & (idx_diff_sec != 60)

        n_intraday_gaps = int(gap_mask.sum())

        if not minute_order_ok:
            bad_minute_order_days.append(day)

        if has_dup_mod:
            duplicate_minute_days.append(day)

        if n_intraday_gaps > 0:
            gap_info = g.loc[gap_mask, ["date", "minute_of_day", "open", "high", "low", "close", "volume"]].copy()
            gap_info["gap_seconds"] = idx_diff_sec[gap_mask].values
            intraday_gap_rows.append(gap_info)

        daily_stats.append({
            "date": day,
            "n_rows": len(g),
            "index_monotonic": bool(idx_mono),
            "minute_of_day_monotonic": minute_order_ok,
            "n_duplicate_minute_of_day": int(dup_mod),
            "n_intraday_gaps_not_60s": n_intraday_gaps,
            "minute_min": int(g["minute_of_day"].min()),
            "minute_max": int(g["minute_of_day"].max()),
        })

        if not idx_mono or not minute_order_ok or has_dup_mod:
            result["ok"] = False

    daily_stats_df = pd.DataFrame(daily_stats)

    result["artifacts"]["daily_stats"] = daily_stats_df
    result["summary"]["n_days"] = int(daily_stats_df.shape[0])
    result["summary"]["days_with_bad_minute_order"] = int(len(bad_minute_order_days))
    result["summary"]["days_with_duplicate_minute_of_day"] = int(len(duplicate_minute_days))
    result["summary"]["days_with_intraday_gaps_not_60s"] = int((daily_stats_df["n_intraday_gaps_not_60s"] > 0).sum())

    result["checks"]["all_days_have_monotonic_minute_of_day"] = (len(bad_minute_order_days) == 0)
    result["checks"]["no_duplicate_minute_of_day_within_day"] = (len(duplicate_minute_days) == 0)
    result["checks"]["all_intraday_steps_are_60s_within_day"] = bool(
        (daily_stats_df["n_intraday_gaps_not_60s"] == 0).all()
    )

    result["artifacts"]["bad_minute_order_days"] = bad_minute_order_days
    result["artifacts"]["duplicate_minute_days"] = duplicate_minute_days

    if intraday_gap_rows:
        result["artifacts"]["intraday_gaps_head"] = pd.concat(intraday_gap_rows, axis=0).head(50)
    else:
        result["artifacts"]["intraday_gaps_head"] = pd.DataFrame()

    # ------------------------------------------------------------
    # 5) Resumen global
    # ------------------------------------------------------------
    result["summary"]["n_rows"] = int(len(df))
    result["summary"]["start"] = df.index.min()
    result["summary"]["end"] = df.index.max()

    # ------------------------------------------------------------
    # 6) Reporte por pantalla
    # ------------------------------------------------------------
    if verbose:
        print("=" * 70)
        print("VALIDACIÓN TEMPORAL DE mnq_intraday")
        print("=" * 70)
        print(f"Rows                     : {result['summary']['n_rows']}")
        print(f"Days                     : {result['summary']['n_days']}")
        print(f"Start                    : {result['summary']['start']}")
        print(f"End                      : {result['summary']['end']}")
        print("-" * 70)
        print(f"Index monotonic          : {result['checks']['index_is_monotonic_increasing']}")
        print(f"Index unique             : {result['checks']['index_is_unique']}")
        print(f"Date == index.date       : {result['checks']['date_column_matches_index_date']}")
        print(f"Global diffs > 0         : {result['checks']['all_global_time_diffs_positive']}")
        print(f"minute_of_day monotonic  : {result['checks']['all_days_have_monotonic_minute_of_day']}")
        print(f"No dup minute_of_day     : {result['checks']['no_duplicate_minute_of_day_within_day']}")
        print(f"Intraday steps = 60s     : {result['checks']['all_intraday_steps_are_60s_within_day']}")
        print("-" * 70)
        print(f"Duplicated timestamps    : {result['summary']['n_duplicated_timestamps']}")
        print(f"Date mismatches          : {result['summary']['n_date_mismatches']}")
        print(f"Non-positive global diffs: {result['summary']['n_non_positive_global_diffs']}")
        print(f"Bad minute order days    : {result['summary']['days_with_bad_minute_order']}")
        print(f"Dup minute_of_day days   : {result['summary']['days_with_duplicate_minute_of_day']}")
        print(f"Days with !=60s gaps     : {result['summary']['days_with_intraday_gaps_not_60s']}")
        print("-" * 70)
        print(f"DATASET OK               : {result['ok']}")
        print("=" * 70)

        if result["summary"]["n_duplicated_timestamps"] > 0:
            print("\nDuplicated timestamps (head):")
            print(pd.Index(result["artifacts"]["duplicated_timestamps"][:10]))

        if result["summary"]["n_date_mismatches"] > 0:
            print("\nDate mismatches (head):")
            print(result["artifacts"]["date_mismatches_head"])

        if result["summary"]["days_with_bad_minute_order"] > 0:
            print("\nDays with bad minute_of_day order (head):")
            print(result["artifacts"]["bad_minute_order_days"][:10])

        if result["summary"]["days_with_duplicate_minute_of_day"] > 0:
            print("\nDays with duplicate minute_of_day (head):")
            print(result["artifacts"]["duplicate_minute_days"][:10])

        if not result["artifacts"]["intraday_gaps_head"].empty:
            print("\nIntraday gaps != 60 seconds (head):")
            print(result["artifacts"]["intraday_gaps_head"])

    return result


# ============================================================
# Ejecución inmediata
# ============================================================
validation = validate_mnq_intraday(mnq_intraday, verbose=True)

# Si quiere abortar automáticamente cuando haya problemas críticos:
critical_checks = [
    "index_is_monotonic_increasing",
    "index_is_unique",
    "date_column_matches_index_date",
    "all_global_time_diffs_positive",
    "all_days_have_monotonic_minute_of_day",
    "no_duplicate_minute_of_day_within_day",
]

failed_critical = [k for k in critical_checks if not validation["checks"].get(k, False)]

if failed_critical:
    raise ValueError(
        "Validación temporal fallida. Checks críticos con error: "
        + ", ".join(failed_critical)
    )

VALIDACIÓN TEMPORAL DE mnq_intraday
Rows                     : 894845
Days                     : 1295
Start                    : 2020-01-02 04:30:00-05:00
End                      : 2025-06-13 16:00:00-04:00
----------------------------------------------------------------------
Index monotonic          : True
Index unique             : True
Date == index.date       : True
Global diffs > 0         : True
minute_of_day monotonic  : True
No dup minute_of_day     : True
Intraday steps = 60s     : True
----------------------------------------------------------------------
Duplicated timestamps    : 0
Date mismatches          : 0
Non-positive global diffs: 0
Bad minute order days    : 0
Dup minute_of_day days   : 0
Days with !=60s gaps     : 0
----------------------------------------------------------------------
DATASET OK               : True


#**2. Agregado de variables temporaltes**

In [34]:
import numpy as np
import pandas as pd


def build_intraday_time_structure_pipeline(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    date_col: str = "date",
    regime_col: str = "regime_id",
    sort_index_if_needed: bool = True,
    drop_duplicate_index: bool = False,
    verbose: bool = True,
):
    """
    Pipeline para construir la estructura temporal intradía mínima del dataset.

    Este pipeline:
    1) valida que el índice sea DatetimeIndex
    2) ordena cronológicamente por índice si es necesario
    3) opcionalmente elimina índices duplicados
    4) agrega la columna minute_of_day
    5) agrega la columna date
    6) agrega la columna regime_id

    Codificación de regime_id
    -------------------------
    0 = overnight
    1 = premarket
    2 = opening
    3 = regular
    4 = closing

    Regímenes intradía (hora NY, intervalos [inicio, fin))
    ------------------------------------------------------
    - premarket : 08:30 <= t < 09:30
    - opening   : 09:30 <= t < 10:30
    - regular   : 10:30 <= t < 15:30
    - closing   : 15:30 <= t < 16:00
    - overnight : resto

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame de entrada con índice datetime.
    minute_col : str, default="minute_of_day"
        Nombre de la columna de minuto del día.
    date_col : str, default="date"
        Nombre de la columna de fecha.
    regime_col : str, default="regime_id"
        Nombre de la columna de régimen.
    sort_index_if_needed : bool, default=True
        Si True, ordena el DataFrame por índice si no está ordenado.
    drop_duplicate_index : bool, default=False
        Si True, elimina duplicados de índice conservando la primera ocurrencia.
    verbose : bool, default=True
        Si True, imprime reporte final.

    Retorna
    -------
    out : pd.DataFrame
        DataFrame con columnas agregadas.
    summary : dict
        Resumen estructurado del proceso.
    """

    regime_name_map = {
        0: "overnight",
        1: "premarket",
        2: "opening",
        3: "regular",
        4: "closing",
    }

    # ---------------------------------------------------------------------
    # Subfunción 1: validaciones básicas
    # ---------------------------------------------------------------------
    def _validate_input(data: pd.DataFrame) -> None:
        if data.empty:
            raise ValueError("El DataFrame está vacío.")

        if not isinstance(data.index, pd.DatetimeIndex):
            raise TypeError("El DataFrame debe tener un DatetimeIndex.")

    # ---------------------------------------------------------------------
    # Subfunción 2: normalización y orden cronológico
    # ---------------------------------------------------------------------
    def _normalize_index(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()

        # Asegurar DatetimeIndex consistente
        out_local.index = pd.to_datetime(out_local.index)

        # Orden cronológico
        if not out_local.index.is_monotonic_increasing:
            if sort_index_if_needed:
                out_local = out_local.sort_index().copy()
            else:
                raise ValueError("El índice datetime no está ordenado crecientemente.")

        # Duplicados de índice
        n_dup_idx = int(out_local.index.duplicated().sum())
        if n_dup_idx > 0:
            if drop_duplicate_index:
                out_local = out_local.loc[~out_local.index.duplicated(keep="first")].copy()
            else:
                raise ValueError(
                    f"Se encontraron {n_dup_idx} timestamps duplicados en el índice."
                )

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 3: minute_of_day
    # ---------------------------------------------------------------------
    def _add_minute_of_day(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()
        out_local[minute_col] = out_local.index.hour * 60 + out_local.index.minute
        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 4: date
    # ---------------------------------------------------------------------
    def _add_date_column(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()
        out_local[date_col] = out_local.index.date

        # Llevar date al frente
        front_cols = [date_col]
        other_cols = [c for c in out_local.columns if c not in front_cols]
        out_local = out_local[front_cols + other_cols]

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 5: regime_id
    # ---------------------------------------------------------------------
    def _add_market_regime(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()

        if minute_col not in out_local.columns:
            raise ValueError(f"No existe la columna {minute_col}.")

        m = out_local[minute_col]

        out_local[regime_col] = np.select(
            [
                (m >= 510) & (m < 570),   # premarket
                (m >= 570) & (m < 630),   # opening
                (m >= 630) & (m < 930),   # regular
                (m >= 930) & (m < 960),   # closing
            ],
            [
                1,  # premarket
                2,  # opening
                3,  # regular
                4,  # closing
            ],
            default=0,  # overnight
        ).astype(np.int8)

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 6: validaciones finales del pipeline
    # ---------------------------------------------------------------------
    def _validate_output(data: pd.DataFrame) -> None:
        required_cols = [date_col, minute_col, regime_col]
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"Faltan columnas de salida esperadas: {missing}")

        if not data.index.is_monotonic_increasing:
            raise ValueError("El output final no quedó ordenado cronológicamente.")

        if data[minute_col].isna().any():
            raise ValueError(f"La columna {minute_col} contiene NaN.")

        if data[regime_col].isna().any():
            raise ValueError(f"La columna {regime_col} contiene NaN.")

        invalid_minutes = (~data[minute_col].between(0, 1439)).sum()
        if invalid_minutes > 0:
            raise ValueError(
                f"Se encontraron {invalid_minutes} valores inválidos en {minute_col}."
            )

        valid_regimes = {0, 1, 2, 3, 4}
        invalid_regimes = (~data[regime_col].isin(valid_regimes)).sum()
        if invalid_regimes > 0:
            raise ValueError(
                f"Se encontraron {invalid_regimes} valores inválidos en {regime_col}."
            )

    # ---------------------------------------------------------------------
    # Subfunción 7: resumen
    # ---------------------------------------------------------------------
    def _build_summary(original_df: pd.DataFrame, final_df: pd.DataFrame) -> dict:
        regime_counts = final_df[regime_col].value_counts(dropna=False).sort_index().to_dict()
        regime_distribution = {}

        total_rows = len(final_df)

        for rid in sorted(regime_counts.keys()):
            count = int(regime_counts[rid])
            regime_distribution[int(rid)] = {
                "name": regime_name_map.get(int(rid), "unknown"),
                "count": count,
                "pct": float(count / total_rows) if total_rows > 0 else np.nan,
            }

        summary_local = {
            "n_rows_input": int(len(original_df)),
            "n_rows_output": int(len(final_df)),
            "n_columns_output": int(final_df.shape[1]),
            "index_is_datetime": isinstance(final_df.index, pd.DatetimeIndex),
            "index_is_sorted": bool(final_df.index.is_monotonic_increasing),
            "n_duplicate_index": int(final_df.index.duplicated().sum()),
            "n_sessions": int(final_df[date_col].nunique()),
            "first_timestamp": str(final_df.index.min()),
            "last_timestamp": str(final_df.index.max()),
            "minute_col": minute_col,
            "date_col": date_col,
            "regime_col": regime_col,
            "minute_min": int(final_df[minute_col].min()),
            "minute_max": int(final_df[minute_col].max()),
            "regime_distribution": regime_distribution,
        }

        return summary_local

    # ---------------------------------------------------------------------
    # Subfunción 8: reporte
    # ---------------------------------------------------------------------
    def _print_report(summary_local: dict) -> None:
        print("=" * 100)
        print("REPORTE | INTRADAY TIME STRUCTURE PIPELINE")
        print("=" * 100)
        print(f"Filas de entrada         : {summary_local['n_rows_input']:,}")
        print(f"Filas de salida          : {summary_local['n_rows_output']:,}")
        print(f"Columnas de salida       : {summary_local['n_columns_output']:,}")
        print(f"Índice datetime          : {summary_local['index_is_datetime']}")
        print(f"Índice ordenado          : {summary_local['index_is_sorted']}")
        print(f"Duplicados en índice     : {summary_local['n_duplicate_index']:,}")
        print(f"Sesiones únicas          : {summary_local['n_sessions']:,}")
        print(f"Primer timestamp         : {summary_local['first_timestamp']}")
        print(f"Último timestamp         : {summary_local['last_timestamp']}")
        print(f"Rango {summary_local['minute_col']}     : {summary_local['minute_min']} - {summary_local['minute_max']}")

        print("-" * 100)
        print("DISTRIBUCIÓN DE REGÍMENES")
        print("-" * 100)

        for rid, info in summary_local["regime_distribution"].items():
            print(
                f"regime_id = {rid} | "
                f"{info['name']:<10} | "
                f"count = {info['count']:,} | "
                f"pct = {info['pct']:.2%}"
            )

        print("=" * 100)

    # ---------------------------------------------------------------------
    # Ejecución principal
    # ---------------------------------------------------------------------
    _validate_input(df)

    out = _normalize_index(df)
    out = _add_minute_of_day(out)
    out = _add_date_column(out)
    out = _add_market_regime(out)

    _validate_output(out)

    summary = _build_summary(df, out)

    if verbose:
        _print_report(summary)

    return out, summary

In [35]:
mnq_intraday, time_summary = build_intraday_time_structure_pipeline(
    mnq_intraday,
    minute_col="minute_of_day",
    date_col="date",
    regime_col="regime_id",
    sort_index_if_needed=True,
    drop_duplicate_index=False,
    verbose=True,
)

REPORTE | INTRADAY TIME STRUCTURE PIPELINE
Filas de entrada         : 894,845
Filas de salida          : 894,845
Columnas de salida       : 8
Índice datetime          : True
Índice ordenado          : True
Duplicados en índice     : 0
Sesiones únicas          : 1,295
Primer timestamp         : 2020-01-02 04:30:00-05:00
Último timestamp         : 2025-06-13 16:00:00-04:00
Rango minute_of_day     : 270 - 960
----------------------------------------------------------------------------------------------------
DISTRIBUCIÓN DE REGÍMENES
----------------------------------------------------------------------------------------------------
regime_id = 0 | overnight  | count = 312,095 | pct = 34.88%
regime_id = 1 | premarket  | count = 77,700 | pct = 8.68%
regime_id = 2 | opening    | count = 77,700 | pct = 8.68%
regime_id = 3 | regular    | count = 388,500 | pct = 43.42%
regime_id = 4 | closing    | count = 38,850 | pct = 4.34%


In [36]:
mnq_intraday

,date,open,high,low,close,volume,minute_of_day,regime_id
datetime,,,,,,,,
2020-01-02 04:30:00-05:00,2020-01-02,8813.25,8813.25,8812.50,8813.25,13,270,0
2020-01-02 04:31:00-05:00,2020-01-02,8812.50,8812.50,8811.00,8812.50,121,271,0
2020-01-02 04:32:00-05:00,2020-01-02,8812.50,8813.25,8811.75,8811.75,53,272,0
2020-01-02 04:33:00-05:00,2020-01-02,8812.00,8812.25,8810.50,8810.50,37,273,0
2020-01-02 04:34:00-05:00,2020-01-02,8810.75,8812.25,8810.50,8812.00,36,274,0
...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,956,4
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,957,4
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,958,4


# **3. Indicadores Técnicos**

## **3.1. Función para cálculo de Indicadores técnicos**

In [37]:
import numpy as np
import pandas as pd

from ta.momentum import ROCIndicator, StochasticOscillator
from ta.volatility import AverageTrueRange


def build_technical_indicators_pipeline(
    df: pd.DataFrame,
    *,
    target: str = "close",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    high_col: str = "high",
    low_col: str = "low",
    validate_order: bool = True,
    drop_na_indicator_rows: bool = False,
    verbose: bool = True,
):
    """
    Pipeline completo para calcular indicadores técnicos intradía por jornada.

    Indicadores calculados
    ----------------------
    - roc_30
    - roc_60
    - stoch_k_30
    - atr_norm_10

    Flujo
    -----
    1) Valida columnas requeridas
    2) Verifica que el índice sea DatetimeIndex
    3) Verifica orden cronológico global
    4) Verifica orden cronológico dentro de cada jornada
    5) Calcula indicadores por día
    6) Imprime reporte final

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame intradía de entrada.
    target : str, default="close"
        Columna objetivo sobre la cual se calculan ROC y Stochastic.
    date_col : str, default="date"
        Columna que identifica la jornada.
    minute_col : str, default="minute_of_day"
        Columna de orden intradía.
    high_col : str, default="high"
        Columna high.
    low_col : str, default="low"
        Columna low.
    validate_order : bool, default=True
        Si True, verifica orden cronológico y duplicados.
    drop_na_indicator_rows : bool, default=False
        Si True, elimina filas con NaN en cualquiera de los indicadores calculados.
    verbose : bool, default=True
        Si True, imprime reporte final.

    Retorna
    -------
    out : pd.DataFrame
        DataFrame con indicadores agregados.
    summary : dict
        Resumen estructurado del proceso.
    """

    indicator_cols = [
        "roc_30",
        "roc_60",
        "stoch_k_30",
        "atr_norm_10",
    ]

    # ---------------------------------------------------------------------
    # Subfunción 1: validaciones estructurales
    # ---------------------------------------------------------------------
    def _validate_input(data: pd.DataFrame) -> None:
        required_cols = [target, high_col, low_col, date_col, minute_col]
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"Faltan columnas requeridas: {missing}")

        if data.empty:
            raise ValueError("El DataFrame está vacío")

        if not isinstance(data.index, pd.DatetimeIndex):
            raise TypeError("El índice debe ser un pd.DatetimeIndex")

    # ---------------------------------------------------------------------
    # Subfunción 2: validación de orden cronológico global y por jornada
    # ---------------------------------------------------------------------
    def _validate_chronological_order(data: pd.DataFrame) -> None:
        if not data.index.is_monotonic_increasing:
            raise ValueError("El índice datetime no está ordenado crecientemente")

        sorted_index = data.sort_values([date_col, minute_col]).index
        if not data.index.equals(sorted_index):
            raise ValueError(
                f"El DataFrame no está ordenado por [{date_col}, {minute_col}]."
            )

        n_dupes = data.duplicated(subset=[date_col, minute_col]).sum()
        if n_dupes > 0:
            raise ValueError(
                f"Se encontraron {n_dupes} combinaciones duplicadas de "
                f"[{date_col}, {minute_col}]."
            )

        # Verificación adicional por jornada
        for session_date, g in data.groupby(date_col, sort=False):
            if not g.index.is_monotonic_increasing:
                raise ValueError(
                    f"La jornada {session_date} no está ordenada cronológicamente por índice."
                )

            if not g[minute_col].is_monotonic_increasing:
                raise ValueError(
                    f"La jornada {session_date} no está ordenada crecientemente por {minute_col}."
                )

    # ---------------------------------------------------------------------
    # Subfunción 3: cálculo de indicadores por jornada
    # ---------------------------------------------------------------------
    def _apply_indicators_one_day(group: pd.DataFrame) -> pd.DataFrame:
        g = group.copy()

        # ROC 30
        g["roc_30"] = ROCIndicator(
            close=g[target],
            window=30,
        ).roc()

        # ROC 60
        g["roc_60"] = ROCIndicator(
            close=g[target],
            window=60,
        ).roc()

        # Stochastic %K con ventana 30
        stoch_30 = StochasticOscillator(
            high=g[high_col],
            low=g[low_col],
            close=g[target],
            window=30,
            smooth_window=3,
        )
        g["stoch_k_30"] = stoch_30.stoch()

        # ATR normalizado con ventana 10
        atr_10 = AverageTrueRange(
            high=g[high_col],
            low=g[low_col],
            close=g[target],
            window=10,
        )
        g["atr_norm_10"] = atr_10.average_true_range() / g[target]

        return g

    # ---------------------------------------------------------------------
    # Subfunción 4: cálculo completo
    # ---------------------------------------------------------------------
    def _compute_indicators(data: pd.DataFrame) -> pd.DataFrame:
        out_local = (
            data.groupby(date_col, group_keys=False, sort=False)
            .apply(_apply_indicators_one_day)
            .sort_index()
        )
        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 5: construir resumen
    # ---------------------------------------------------------------------
    def _build_summary(original_df: pd.DataFrame, final_df: pd.DataFrame) -> dict:
        summary_local = {
            "n_rows_input": int(len(original_df)),
            "n_rows_output": int(len(final_df)),
            "n_sessions": int(original_df[date_col].nunique()),
            "target_col": target,
            "date_col": date_col,
            "minute_col": minute_col,
            "validate_order": bool(validate_order),
            "drop_na_indicator_rows": bool(drop_na_indicator_rows),
            "indicators": {},
        }

        for col in indicator_cols:
            n_nan = int(final_df[col].isna().sum())
            n_valid = int(final_df[col].notna().sum())

            if n_valid > 0:
                desc = final_df[col].dropna().describe()
                summary_local["indicators"][col] = {
                    "n_valid": n_valid,
                    "n_nan": n_nan,
                    "mean": float(desc["mean"]),
                    "std": float(desc["std"]) if "std" in desc else np.nan,
                    "min": float(desc["min"]),
                    "p25": float(desc["25%"]),
                    "p50": float(desc["50%"]),
                    "p75": float(desc["75%"]),
                    "max": float(desc["max"]),
                }
            else:
                summary_local["indicators"][col] = {
                    "n_valid": 0,
                    "n_nan": n_nan,
                    "mean": np.nan,
                    "std": np.nan,
                    "min": np.nan,
                    "p25": np.nan,
                    "p50": np.nan,
                    "p75": np.nan,
                    "max": np.nan,
                }

        return summary_local

    # ---------------------------------------------------------------------
    # Subfunción 6: impresión de reporte
    # ---------------------------------------------------------------------
    def _print_report(summary_local: dict) -> None:
        print("=" * 100)
        print("REPORTE | TECHNICAL INDICATORS PIPELINE")
        print("=" * 100)
        print(f"Filas de entrada         : {summary_local['n_rows_input']:,}")
        print(f"Filas de salida          : {summary_local['n_rows_output']:,}")
        print(f"Sesiones únicas          : {summary_local['n_sessions']:,}")
        print(f"Columna target           : {summary_local['target_col']}")
        print(f"Columna fecha            : {summary_local['date_col']}")
        print(f"Columna orden intradía   : {summary_local['minute_col']}")
        print(f"Validar orden            : {summary_local['validate_order']}")
        print(f"Drop filas con NaN       : {summary_local['drop_na_indicator_rows']}")

        print("-" * 100)
        print("RESUMEN POR INDICADOR")
        print("-" * 100)

        for col in indicator_cols:
            s = summary_local["indicators"][col]
            print(f"\nIndicador                : {col}")
            print(f"  Observaciones válidas  : {s['n_valid']:,}")
            print(f"  Observaciones NaN      : {s['n_nan']:,}")

            if s["n_valid"] > 0:
                print(f"  Mean                   : {s['mean']:.6f}")
                print(f"  Std                    : {s['std']:.6f}")
                print(f"  Min                    : {s['min']:.6f}")
                print(f"  P25                    : {s['p25']:.6f}")
                print(f"  P50                    : {s['p50']:.6f}")
                print(f"  P75                    : {s['p75']:.6f}")
                print(f"  Max                    : {s['max']:.6f}")
            else:
                print("  No hay valores válidos para este indicador.")

        print("=" * 100)

    # ---------------------------------------------------------------------
    # Ejecución principal
    # ---------------------------------------------------------------------
    _validate_input(df)

    if validate_order:
        _validate_chronological_order(df)

    out = _compute_indicators(df)

    if drop_na_indicator_rows:
        out = out.dropna(subset=indicator_cols).copy()

    summary = _build_summary(df, out)

    if verbose:
        _print_report(summary)

    return out, summary

## **3.2. Cálculo de indicadores técnicos**

In [38]:
mnq_features, tech_summary = build_technical_indicators_pipeline(
    mnq_intraday,
    target="close",
    date_col="date",
    minute_col="minute_of_day",
    high_col="high",
    low_col="low",
    validate_order=True,
    drop_na_indicator_rows=True,
    verbose=True,
)

REPORTE | TECHNICAL INDICATORS PIPELINE
Filas de entrada         : 894,845
Filas de salida          : 817,145
Sesiones únicas          : 1,295
Columna target           : close
Columna fecha            : date
Columna orden intradía   : minute_of_day
Validar orden            : True
Drop filas con NaN       : True
----------------------------------------------------------------------------------------------------
RESUMEN POR INDICADOR
----------------------------------------------------------------------------------------------------

Indicador                : roc_30
  Observaciones válidas  : 817,145
  Observaciones NaN      : 0
  Mean                   : 0.002336
  Std                    : 0.281450
  Min                    : -5.197729
  P25                    : -0.105952
  P50                    : 0.008586
  P75                    : 0.119323
  Max                    : 8.558820

Indicador                : roc_60
  Observaciones válidas  : 817,145
  Observaciones NaN      : 0
  Mean     

In [39]:
assert not mnq_features.isna().any().any(), \
    "El dataset contiene NaN"

In [41]:
mnq_features

,date,open,high,low,close,volume,minute_of_day,regime_id,roc_30,roc_60,stoch_k_30,atr_norm_10
datetime,,,,,,,,,,,,
2020-01-02 05:30:00-05:00,2020-01-02,8819.50,8819.75,8818.75,8819.25,83,330,0,0.039702,0.068079,90.476190,0.000115
2020-01-02 05:31:00-05:00,2020-01-02,8819.25,8819.75,8818.50,8818.75,55,331,0,0.034030,0.070922,80.952381,0.000118
2020-01-02 05:32:00-05:00,2020-01-02,8818.50,8819.00,8818.25,8819.00,36,332,0,0.017012,0.082277,85.714286,0.000115
2020-01-02 05:33:00-05:00,2020-01-02,8819.00,8819.00,8818.25,8818.25,28,333,0,0.025522,0.087963,71.428571,0.000112
2020-01-02 05:34:00-05:00,2020-01-02,8818.25,8819.00,8818.00,8818.75,27,334,0,0.031193,0.076600,80.952381,0.000112
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,956,4,-0.055480,-0.203125,22.413793,0.000888
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,957,4,-0.025429,-0.182336,31.034483,0.000889
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,958,4,-0.024275,-0.102800,28.275862,0.000876


# **4. Construcción de targets**

##**4.1. Construcción de targets T2**



In [48]:
import numpy as np
import pandas as pd


def build_t2_threshold_targets_pipeline(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    horizons: tuple[int, ...] = (90, 120),
    percentile: float = 70,
    drop_na_targets: bool = True,
    validate_order: bool = True,
    verbose: bool = True,
):
    """
    Pipeline completo para construir targets T2 con umbral basado en percentiles.

    Flujo:
    1) Calcula close_fwd_h
    2) Calcula delta_h = close_fwd_h - close_t
    3) Calcula threshold_h = percentile(|delta_h|)
    4) Construye target ternario:
           1  si delta_h > +threshold_h
           0  si -threshold_h <= delta_h <= +threshold_h
          -1  si delta_h < -threshold_h
    5) Elimina NaN al final de cada jornada, generados por falta de futuro

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame intradía.
    close_col : str
        Nombre de la columna de cierre.
    date_col : str
        Nombre de la columna de fecha/sesión.
    minute_col : str
        Nombre de la columna de orden intradía.
    horizons : tuple[int, ...]
        Horizontes a procesar.
    percentile : float
        Percentil usado para calcular thresholds sobre |delta_h|.
    drop_na_targets : bool
        Si True, elimina filas finales inválidas por jornada donde los targets son NaN.
    validate_order : bool
        Si True, valida orden temporal y duplicados por día/minuto.
    verbose : bool
        Si True, imprime informe final.

    Retorna
    -------
    out : pd.DataFrame
        Copia del DataFrame con nuevas columnas:
        - close_fwd_{h}
        - delta_{h}
        - t2_dir_thr_{h}
    thresholds : dict
        Diccionario con thresholds por horizonte.
    """

    # ---------------------------------------------------------------------
    # Subfunción 1: validaciones básicas
    # ---------------------------------------------------------------------
    def _validate_input(data: pd.DataFrame):
        required_cols = [close_col, date_col, minute_col]
        missing = [c for c in required_cols if c not in data.columns]
        if missing:
            raise ValueError(f"Faltan columnas requeridas: {missing}")

        if data.empty:
            raise ValueError("El DataFrame está vacío.")

        if validate_order:
            sorted_df = data.sort_values([date_col, minute_col])
            if not data.index.equals(sorted_df.index):
                raise ValueError(
                    f"El DataFrame no está ordenado por [{date_col}, {minute_col}]."
                )

            n_dupes = data.duplicated(subset=[date_col, minute_col]).sum()
            if n_dupes > 0:
                raise ValueError(
                    f"Se encontraron {n_dupes} combinaciones duplicadas de "
                    f"[{date_col}, {minute_col}]."
                )

    # ---------------------------------------------------------------------
    # Subfunción 2: cálculo de precios forward y deltas
    # ---------------------------------------------------------------------
    def _compute_forward_and_delta(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()
        grouped_close = out_local.groupby(date_col, sort=False)[close_col]

        for h in horizons:
            fwd_col = f"close_fwd_{h}"
            delta_col = f"delta_{h}"

            out_local[fwd_col] = grouped_close.shift(-h)
            out_local[delta_col] = out_local[fwd_col] - out_local[close_col]

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 3: cálculo de thresholds desde percentiles de |delta_h|
    # ---------------------------------------------------------------------
    def _compute_thresholds(data: pd.DataFrame) -> dict:
        thresholds_local = {}

        for h in horizons:
            delta_col = f"delta_{h}"

            if delta_col not in data.columns:
                raise ValueError(f"No existe {delta_col} en el DataFrame")

            abs_delta = data[delta_col].abs().dropna()

            if len(abs_delta) == 0:
                raise ValueError(
                    f"No hay valores válidos en {delta_col} para calcular threshold."
                )

            thr = np.percentile(abs_delta, percentile)
            thresholds_local[h] = float(thr)

        return thresholds_local

    # ---------------------------------------------------------------------
    # Subfunción 4: construcción del target ternario
    # ---------------------------------------------------------------------
    def _build_targets(data: pd.DataFrame, thresholds_local: dict) -> pd.DataFrame:
        out_local = data.copy()

        for h in horizons:
            fwd_col = f"close_fwd_{h}"
            delta_col = f"delta_{h}"
            target_col = f"t2_dir_thr_{h}"
            thr = thresholds_local[h]

            delta = out_local[delta_col]

            out_local[target_col] = np.where(
                out_local[fwd_col].isna(),
                np.nan,
                np.where(
                    delta > thr,
                    1,
                    np.where(delta < -thr, -1, 0)
                )
            ).astype("float")

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 5: eliminar NaN al final de cada jornada
    # ---------------------------------------------------------------------
    def _drop_trailing_nans_per_day(data: pd.DataFrame) -> pd.DataFrame:
        out_local = data.copy()
        target_cols = [f"t2_dir_thr_{h}" for h in horizons]

        def trim_group(g: pd.DataFrame) -> pd.DataFrame:
            valid_mask = g[target_cols].notna().all(axis=1)

            if not valid_mask.any():
                return g.iloc[0:0].copy()

            last_valid_pos = np.where(valid_mask.to_numpy())[0][-1]
            return g.iloc[: last_valid_pos + 1].copy()

        out_local = (
            out_local.groupby(date_col, group_keys=False, sort=False)
            .apply(trim_group)
        )

        return out_local

    # ---------------------------------------------------------------------
    # Subfunción 6: impresión de informe
    # ---------------------------------------------------------------------
    def _print_report(original_df: pd.DataFrame, final_df: pd.DataFrame, thresholds_local: dict):
        print("=" * 100)
        print("REPORTE | PIPELINE T2 THRESHOLD TARGET")
        print("=" * 100)
        print(f"Filas de entrada         : {len(original_df):,}")
        print(f"Filas de salida          : {len(final_df):,}")
        print(f"Horizontes               : {horizons}")
        print(f"Percentil usado          : {percentile}")
        print(f"Drop NA targets          : {drop_na_targets}")

        print("-" * 100)
        print("THRESHOLDS ESTIMADOS")
        print("-" * 100)
        for h in horizons:
            thr = thresholds_local[h]
            print(f"h = {h:>3} | threshold = {thr:>8.2f} pts | ≈ {thr * 2:>8.2f} USD")

        print("-" * 100)
        print("RESUMEN POR HORIZONTE")
        print("-" * 100)

        for h in horizons:
            fwd_col = f"close_fwd_{h}"
            delta_col = f"delta_{h}"
            target_col = f"t2_dir_thr_{h}"
            thr = thresholds_local[h]

            valid_delta = final_df[delta_col].dropna()
            valid_target = final_df[target_col].dropna()

            n_valid = len(valid_target)
            n_up = (final_df[target_col] == 1).sum()
            n_mid = (final_df[target_col] == 0).sum()
            n_down = (final_df[target_col] == -1).sum()

            print(f"\nHorizonte h = {h}")
            print(f"  Columna forward        : {fwd_col}")
            print(f"  Columna delta          : {delta_col}")
            print(f"  Columna target         : {target_col}")
            print(f"  Threshold              : {thr:.2f} pts")
            print(f"  Observaciones válidas  : {n_valid:,}")

            if len(valid_delta) > 0:
                print(f"  Mean(|delta|)          : {valid_delta.abs().mean():.2f}")
                print(f"  Median(|delta|)        : {valid_delta.abs().median():.2f}")
                print(f"  Max(|delta|)           : {valid_delta.abs().max():.2f}")

            if n_valid > 0:
                print(f"  Clase  1 (sube)        : {n_up:,} ({n_up/n_valid:.2%})")
                print(f"  Clase  0 (neutral)     : {n_mid:,} ({n_mid/n_valid:.2%})")
                print(f"  Clase -1 (baja)        : {n_down:,} ({n_down/n_valid:.2%})")
            else:
                print("  No hay observaciones válidas para el target.")

        print("=" * 100)

    # ---------------------------------------------------------------------
    # Ejecución principal
    # ---------------------------------------------------------------------
    _validate_input(df)

    out = _compute_forward_and_delta(df)
    thresholds = _compute_thresholds(out)
    out = _build_targets(out, thresholds)

    if drop_na_targets:
        out = _drop_trailing_nans_per_day(out)

    if verbose:
        _print_report(df, out, thresholds)

    return out, thresholds

## **4.2. Aplicación**

In [58]:
mnq_t2, thresholds_t2 = build_t2_threshold_targets_pipeline(
    mnq_features,
    close_col="close",
    date_col="date",
    minute_col="minute_of_day",
    horizons=(90, 120),
    percentile=70,
    drop_na_targets=True,
    validate_order=True,
    verbose=True,
)

REPORTE | PIPELINE T2 THRESHOLD TARGET
Filas de entrada         : 817,145
Filas de salida          : 661,745
Horizontes               : (90, 120)
Percentil usado          : 70
Drop NA targets          : True
----------------------------------------------------------------------------------------------------
THRESHOLDS ESTIMADOS
----------------------------------------------------------------------------------------------------
h =  90 | threshold =    54.75 pts | ≈   109.50 USD
h = 120 | threshold =    65.75 pts | ≈   131.50 USD
----------------------------------------------------------------------------------------------------
RESUMEN POR HORIZONTE
----------------------------------------------------------------------------------------------------

Horizonte h = 90
  Columna forward        : close_fwd_90
  Columna delta          : delta_90
  Columna target         : t2_dir_thr_90
  Threshold              : 54.75 pts
  Observaciones válidas  : 661,745
  Mean(|delta|)          : 47.56
 

# **5. Verificar existencia de señal**

## **5.1. Creación de split temporales**

In [60]:
import pandas as pd


def time_based_split_by_date(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    train_ratio: float = 0.70,
    valid_ratio: float = 0.15,
    test_ratio: float = 0.15,
    verbose: bool = True,
):
    """
    Realiza un split temporal (train/valid/test) basado en fechas.

    - No mezcla días
    - Respeta orden cronológico
    - Evita leakage

    Retorna
    -------
    train_df, valid_df, test_df
    """

    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-6:
        raise ValueError("Los ratios deben sumar 1.0")

    # Ordenar por fecha y minuto (por seguridad)
    df_sorted = df.sort_values([date_col, "minute_of_day"]).copy()

    # Fechas únicas ordenadas
    unique_dates = df_sorted[date_col].drop_duplicates().sort_values().values
    n_dates = len(unique_dates)

    # Índices de corte
    train_end = int(n_dates * train_ratio)
    valid_end = train_end + int(n_dates * valid_ratio)

    # Fechas por split
    train_dates = unique_dates[:train_end]
    valid_dates = unique_dates[train_end:valid_end]
    test_dates  = unique_dates[valid_end:]

    # Crear splits
    train_df = df_sorted[df_sorted[date_col].isin(train_dates)].copy()
    valid_df = df_sorted[df_sorted[date_col].isin(valid_dates)].copy()
    test_df  = df_sorted[df_sorted[date_col].isin(test_dates)].copy()

    if verbose:
        print("=" * 80)
        print("TIME SPLIT SUMMARY")
        print("=" * 80)

        print(f"Total días        : {n_dates}")
        print(f"Train días        : {len(train_dates)} ({len(train_dates)/n_dates:.2%})")
        print(f"Valid días        : {len(valid_dates)} ({len(valid_dates)/n_dates:.2%})")
        print(f"Test días         : {len(test_dates)} ({len(test_dates)/n_dates:.2%})")

        print("-" * 80)

        print(f"Train fechas      : {train_dates[0]} → {train_dates[-1]}")
        print(f"Valid fechas      : {valid_dates[0]} → {valid_dates[-1]}")
        print(f"Test fechas       : {test_dates[0]} → {test_dates[-1]}")

        print("-" * 80)

        print(f"Train filas       : {len(train_df):,}")
        print(f"Valid filas       : {len(valid_df):,}")
        print(f"Test filas        : {len(test_df):,}")

        print("=" * 80)

    return train_df, valid_df, test_df

In [61]:
mnq_train, mnq_valid, mnq_test = time_based_split_by_date(
    mnq_t2,
    date_col="date",
    train_ratio=0.70,
    valid_ratio=0.15,
    test_ratio=0.15,
    verbose=True,
)

TIME SPLIT SUMMARY
Total días        : 1295
Train días        : 906 (69.96%)
Valid días        : 194 (14.98%)
Test días         : 195 (15.06%)
--------------------------------------------------------------------------------
Train fechas      : 2020-01-02 → 2023-10-30
Valid fechas      : 2023-10-31 → 2024-08-21
Test fechas       : 2024-08-22 → 2025-06-13
--------------------------------------------------------------------------------
Train filas       : 462,966
Valid filas       : 99,134
Test filas        : 99,645


## **5.2. Evaluación multiclass balanceado**

In [62]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


def evaluate_multiclass_classifier_balanced(
    *,
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str],
    model_name: str = "logistic",
    impute_strategy: str = "median",
    scale_features: bool = True,
    dropna_target: bool = True,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """
    Entrena un clasificador multiclase baseline con class_weight="balanced".

    Targets esperados:
        -1, 0, 1

    Naive:
        siempre predice la clase mayoritaria del train

    Métricas:
        - accuracy
        - balanced_accuracy
        - f1_macro
    """

    # -----------------------------
    # Validaciones
    # -----------------------------
    missing_features = [c for c in feature_cols if c not in train_df.columns]
    if missing_features:
        raise ValueError(f"Faltan features en train_df: {missing_features}")

    for tgt in target_cols:
        for split_name, split_df in {
            "train": train_df,
            "valid": valid_df,
            "test": test_df,
        }.items():
            if tgt not in split_df.columns:
                raise ValueError(f"Falta target '{tgt}' en split '{split_name}'")

    # -----------------------------
    # Modelo
    # -----------------------------
    if model_name == "logistic":
        clf = LogisticRegression(
            max_iter=3000,
            multi_class="auto",
            random_state=42,
            class_weight="balanced",  # 🔥 CAMBIO CLAVE
        )
    elif model_name == "ridge":
        clf = RidgeClassifier(
            random_state=42,
            class_weight="balanced",  # 🔥 CAMBIO CLAVE
        )
    else:
        raise ValueError("model_name debe ser 'logistic' o 'ridge'")

    steps = [("imputer", SimpleImputer(strategy=impute_strategy))]
    if scale_features:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", clf))
    pipeline = Pipeline(steps)

    split_map = {
        "train": train_df,
        "valid": valid_df,
        "test": test_df,
    }

    results = []
    fitted_models = {}

    # -----------------------------
    # Loop por target
    # -----------------------------
    for target_col in target_cols:
        train_work = train_df[feature_cols + [target_col]].copy()
        if dropna_target:
            train_work = train_work[train_work[target_col].notna()].copy()

        X_train = train_work[feature_cols]
        y_train = train_work[target_col].astype(int)

        train_classes = sorted(y_train.unique().tolist())
        if len(train_classes) < 2:
            raise ValueError(
                f"El target '{target_col}' no tiene suficientes clases: {train_classes}"
            )

        majority_class = int(y_train.value_counts().idxmax())

        model = pipeline.fit(X_train, y_train)
        fitted_models[target_col] = {
            "model": model,
            "majority_class": majority_class,
        }

        # -----------------------------
        # Evaluación
        # -----------------------------
        for split_name, split_df in split_map.items():
            work = split_df[feature_cols + [target_col]].copy()
            if dropna_target:
                work = work[work[target_col].notna()].copy()

            X = work[feature_cols]
            y = work[target_col].astype(int)

            y_pred = model.predict(X)
            y_pred_naive = np.full(len(y), majority_class)

            acc_model = accuracy_score(y, y_pred)
            bal_acc_model = balanced_accuracy_score(y, y_pred)
            f1_model = f1_score(y, y_pred, average="macro", zero_division=0)

            acc_naive = accuracy_score(y, y_pred_naive)
            bal_acc_naive = balanced_accuracy_score(y, y_pred_naive)
            f1_naive = f1_score(y, y_pred_naive, average="macro", zero_division=0)

            class_dist = y.value_counts(normalize=True).to_dict()

            results.append(
                {
                    "model": model_name + "_balanced",
                    "target": target_col,
                    "split": split_name,
                    "n_samples": len(y),

                    "pct_-1": class_dist.get(-1, 0.0),
                    "pct_0": class_dist.get(0, 0.0),
                    "pct_1": class_dist.get(1, 0.0),

                    "acc_model": acc_model,
                    "acc_naive": acc_naive,
                    "acc_gain": acc_model - acc_naive,

                    "bal_acc_model": bal_acc_model,
                    "bal_acc_naive": bal_acc_naive,
                    "bal_acc_gain": bal_acc_model - bal_acc_naive,

                    "f1_model": f1_model,
                    "f1_naive": f1_naive,
                    "f1_gain": f1_model - f1_naive,
                }
            )

    results_df = pd.DataFrame(results)

    if verbose:
        print("=" * 120)
        print(f"BASELINE T2 (BALANCED) | model={model_name}")
        print("=" * 120)

        cols_show = [
            "target",
            "split",
            "acc_model",
            "acc_naive",
            "acc_gain",
            "bal_acc_model",
            "bal_acc_naive",
            "bal_acc_gain",
            "f1_model",
            "f1_naive",
            "f1_gain",
        ]

        print(results_df[cols_show].round(4).to_string(index=False))

    return results_df, fitted_models

In [65]:
feature_cols = ['regime_id', 'roc_30', 'roc_60', 'stoch_k_30', 'atr_norm_10']
target_cols = ['t2_dir_thr_90', 't2_dir_thr_120']

results_t2_balanced, models_t2_balanced = evaluate_multiclass_classifier_balanced(
    train_df=mnq_train,
    valid_df=mnq_valid,
    test_df=mnq_test,
    feature_cols=feature_cols,
    target_cols=target_cols,
    model_name="logistic",
    verbose=True,
)

BASELINE T2 (BALANCED) | model=logistic
        target split  acc_model  acc_naive  acc_gain  bal_acc_model  bal_acc_naive  bal_acc_gain  f1_model  f1_naive  f1_gain
 t2_dir_thr_90 train     0.6000     0.7181   -0.1181         0.4315         0.3333        0.0981    0.4186    0.2786   0.1400
 t2_dir_thr_90 valid     0.6982     0.7275   -0.0293         0.4020         0.3333        0.0686    0.4068    0.2808   0.1260
 t2_dir_thr_90  test     0.5607     0.5878   -0.0272         0.4105         0.3333        0.0772    0.4084    0.2468   0.1616
t2_dir_thr_120 train     0.5953     0.7174   -0.1221         0.4215         0.3333        0.0882    0.4070    0.2785   0.1285
t2_dir_thr_120 valid     0.6999     0.7334   -0.0334         0.3847         0.3333        0.0513    0.3872    0.2821   0.1051
t2_dir_thr_120  test     0.5628     0.5885   -0.0257         0.4064         0.3333        0.0731    0.4014    0.2470   0.1544


## **5.3. Conclusión parcial — Evaluación de targets T2 (clasificación multiclase)**

Se realizó una evaluación baseline de los targets `t2_dir_thr_90` y `t2_dir_thr_120` utilizando un modelo lineal balanceado (Logistic Regression), con el objetivo de verificar la existencia de señal predictiva explotable en un esquema de clasificación ternaria (-1, 0, 1).

Los resultados muestran que, si bien la métrica de accuracy es inferior al baseline naive (dominado por la clase mayoritaria), esto no resulta relevante en este contexto debido al desbalance inherente del target. En su lugar, las métricas más informativas son `balanced_accuracy` y `f1_macro`.

En este sentido, ambos targets presentan:

* una mejora consistente en `balanced_accuracy` respecto al baseline (~+0.05 a +0.10),
* una mejora significativa en `f1_macro` (~+0.10 a +0.16),
* estabilidad entre los splits de validación y test,
* ausencia de señales claras de sobreajuste.

Estos resultados indican que el modelo es capaz de capturar cierta estructura en los datos y distinguir entre las tres clases mejor que una estrategia aleatoria o naive, lo que sugiere la presencia de señal predictiva real.

No se observan diferencias relevantes entre los horizontes 90 y 120, mostrando ambos un comportamiento muy similar en términos de desempeño y robustez.

En conjunto, la evidencia sugiere que los targets T2 presentan una señal **débil a moderada pero consistente**, de carácter principalmente direccional, lo cual los hace potencialmente útiles para tareas de clasificación, filtrado de señales o construcción de reglas de decisión en el contexto de trading.

Como siguiente paso, resulta fundamental profundizar el análisis a nivel de régimen de mercado, ya que la señal puede no ser homogénea en el tiempo. Es esperable que ciertos regímenes (por ejemplo, apertura o alta volatilidad) concentren mayor capacidad predictiva, mientras que otros (como overnight o lateralidad) presenten menor señal. Este análisis permitirá identificar dónde el modelo realmente aporta valor y mejorar el diseño de la estrategia.


## **5.4. Análisis por régimen**

In [66]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


def evaluate_multiclass_classifier_by_regime(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    *,
    feature_cols: list[str],
    target_cols: list[str],
    regime_col: str = "regime_id",
    model_name: str = "logistic",
    min_samples_per_class: int = 20,
    min_total_samples: int = 100,
    verbose: bool = True,
):
    """
    Evalúa targets multiclase por régimen de mercado.

    Para cada target y cada regime_id:
    - filtra train/valid/test por régimen
    - entrena un clasificador balanceado en train
    - compara contra baseline naive (clase mayoritaria de train)
    - calcula métricas por split

    Modelos soportados
    ------------------
    - logistic
    - ridge

    Métricas
    --------
    - accuracy
    - balanced_accuracy
    - f1_macro

    Parámetros
    ----------
    train_df, valid_df, test_df : pd.DataFrame
        Splits del dataset.
    feature_cols : list[str]
        Variables predictoras.
    target_cols : list[str]
        Targets multiclase a evaluar.
    regime_col : str, default="regime_id"
        Columna de régimen.
    model_name : str, default="logistic"
        Modelo a utilizar.
    min_samples_per_class : int, default=20
        Mínimo de muestras por clase en train para permitir entrenamiento.
    min_total_samples : int, default=100
        Mínimo de muestras totales en train para permitir entrenamiento.
    verbose : bool, default=True
        Si True, imprime reporte.

    Retorna
    -------
    results_df : pd.DataFrame
        Tabla consolidada de resultados por régimen.
    models_dict : dict
        Modelos entrenados, indexados por (target, regime_id).
    """

    regime_name_map = {
        0: "overnight",
        1: "premarket",
        2: "opening",
        3: "regular",
        4: "closing",
    }

    # ------------------------------------------------------------------
    # Subfunción 1: validaciones
    # ------------------------------------------------------------------
    def _validate_inputs():
        for name, df_ in {
            "train_df": train_df,
            "valid_df": valid_df,
            "test_df": test_df,
        }.items():
            if df_.empty:
                raise ValueError(f"{name} está vacío.")

            missing_feats = [c for c in feature_cols if c not in df_.columns]
            missing_tgts = [c for c in target_cols if c not in df_.columns]
            missing_reg = [regime_col] if regime_col not in df_.columns else []

            missing = missing_feats + missing_tgts + missing_reg
            if missing:
                raise ValueError(f"Faltan columnas en {name}: {missing}")

    # ------------------------------------------------------------------
    # Subfunción 2: modelo
    # ------------------------------------------------------------------
    def _build_model():
        if model_name == "logistic":
            clf = LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                multi_class="auto",
                random_state=42,
            )
        elif model_name == "ridge":
            clf = RidgeClassifier(
                class_weight="balanced",
                random_state=42,
            )
        else:
            raise ValueError("model_name debe ser 'logistic' o 'ridge'")

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", clf),
        ])
        return pipe

    # ------------------------------------------------------------------
    # Subfunción 3: baseline naive
    # ------------------------------------------------------------------
    def _compute_naive_predictions(y_train: pd.Series, n_pred: int):
        majority_class = y_train.value_counts().idxmax()
        return np.full(shape=n_pred, fill_value=majority_class), majority_class

    # ------------------------------------------------------------------
    # Subfunción 4: métricas
    # ------------------------------------------------------------------
    def _compute_metrics(y_true, y_pred):
        return {
            "acc": accuracy_score(y_true, y_pred),
            "bal_acc": balanced_accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        }

    # ------------------------------------------------------------------
    # Subfunción 5: evaluar un target en un régimen
    # ------------------------------------------------------------------
    def _evaluate_one_target_one_regime(target_col: str, regime_id: int):
        cols_needed = feature_cols + [target_col, regime_col]

        tr = train_df.loc[train_df[regime_col] == regime_id, cols_needed].dropna(subset=[target_col]).copy()
        va = valid_df.loc[valid_df[regime_col] == regime_id, cols_needed].dropna(subset=[target_col]).copy()
        te = test_df.loc[test_df[regime_col] == regime_id, cols_needed].dropna(subset=[target_col]).copy()

        if len(tr) < min_total_samples:
            return None, None

        y_train = tr[target_col].astype(int)
        class_counts = y_train.value_counts().sort_index()

        if y_train.nunique() < 2:
            return None, None

        if (class_counts < min_samples_per_class).any():
            return None, None

        X_train = tr[feature_cols]
        X_valid = va[feature_cols]
        X_test = te[feature_cols]

        y_valid = va[target_col].astype(int)
        y_test = te[target_col].astype(int)

        model = _build_model()
        model.fit(X_train, y_train)

        split_data = {
            "train": (X_train, y_train),
            "valid": (X_valid, y_valid),
            "test": (X_test, y_test),
        }

        rows = []
        for split_name, (X_split, y_split) in split_data.items():
            if len(y_split) == 0:
                continue

            y_pred_model = model.predict(X_split)
            y_pred_naive, majority_class = _compute_naive_predictions(y_train, len(y_split))

            m_model = _compute_metrics(y_split, y_pred_model)
            m_naive = _compute_metrics(y_split, y_pred_naive)

            row = {
                "target": target_col,
                "regime_id": regime_id,
                "regime_name": regime_name_map.get(regime_id, f"regime_{regime_id}"),
                "split": split_name,
                "n_samples": int(len(y_split)),
                "n_classes_train": int(y_train.nunique()),
                "majority_class_train": int(majority_class),
                "acc_model": m_model["acc"],
                "acc_naive": m_naive["acc"],
                "acc_gain": m_model["acc"] - m_naive["acc"],
                "bal_acc_model": m_model["bal_acc"],
                "bal_acc_naive": m_naive["bal_acc"],
                "bal_acc_gain": m_model["bal_acc"] - m_naive["bal_acc"],
                "f1_model": m_model["f1_macro"],
                "f1_naive": m_naive["f1_macro"],
                "f1_gain": m_model["f1_macro"] - m_naive["f1_macro"],
                "train_class_counts": dict(class_counts),
            }
            rows.append(row)

        return rows, model

    # ------------------------------------------------------------------
    # Subfunción 6: reporte
    # ------------------------------------------------------------------
    def _print_report(results_df: pd.DataFrame):
        print("=" * 120)
        print(f"BASELINE T2 POR RÉGIMEN | model={model_name}")
        print("=" * 120)

        if results_df.empty:
            print("No se generaron resultados. Revisar tamaño muestral por régimen.")
            print("=" * 120)
            return

        cols_show = [
            "target",
            "regime_name",
            "split",
            "n_samples",
            "bal_acc_model",
            "bal_acc_naive",
            "bal_acc_gain",
            "f1_model",
            "f1_naive",
            "f1_gain",
        ]

        print(results_df[cols_show].to_string(index=False))

        print("-" * 120)
        print("RESUMEN TEST")
        print("-" * 120)

        test_df_ = results_df[results_df["split"] == "test"].copy()
        if not test_df_.empty:
            summary = (
                test_df_
                .groupby(["target", "regime_name"], as_index=False)[["bal_acc_gain", "f1_gain"]]
                .mean()
                .sort_values(["target", "bal_acc_gain", "f1_gain"], ascending=[True, False, False])
            )
            print(summary.to_string(index=False))

        print("=" * 120)

    # ------------------------------------------------------------------
    # Ejecución principal
    # ------------------------------------------------------------------
    _validate_inputs()

    regimes = sorted(set(train_df[regime_col].dropna().astype(int).unique()))
    all_rows = []
    models_dict = {}

    for target_col in target_cols:
        for regime_id in regimes:
            rows, model = _evaluate_one_target_one_regime(target_col, regime_id)

            if rows is None:
                continue

            all_rows.extend(rows)
            models_dict[(target_col, regime_id)] = model

    results_df = pd.DataFrame(all_rows)

    if not results_df.empty:
        results_df = results_df.sort_values(
            by=["target", "regime_id", "split"],
            ascending=[True, True, True]
        ).reset_index(drop=True)

    if verbose:
        _print_report(results_df)

    return results_df, models_dict

In [67]:
results_t2_by_regime, models_t2_by_regime = evaluate_multiclass_classifier_by_regime(
    train_df=mnq_train,
    valid_df=mnq_valid,
    test_df=mnq_test,
    feature_cols=feature_cols,
    target_cols=target_cols,
    regime_col="regime_id",
    model_name="logistic",
    min_samples_per_class=20,
    min_total_samples=100,
    verbose=True,
)

BASELINE T2 POR RÉGIMEN | model=logistic
        target regime_name split  n_samples  bal_acc_model  bal_acc_naive  bal_acc_gain  f1_model  f1_naive  f1_gain
t2_dir_thr_120   overnight  test      35100       0.387625       0.333333      0.054291  0.385112  0.273502 0.111610
t2_dir_thr_120   overnight train     163080       0.430115       0.333333      0.096781  0.394561  0.297709 0.096852
t2_dir_thr_120   overnight valid      34920       0.357024       0.333333      0.023691  0.351227  0.299213 0.052014
t2_dir_thr_120   premarket  test      11700       0.357461       0.333333      0.024127  0.309111  0.171743 0.137368
t2_dir_thr_120   premarket train      54360       0.399794       0.333333      0.066461  0.393125  0.232306 0.160818
t2_dir_thr_120   premarket valid      11640       0.389545       0.333333      0.056212  0.377252  0.240597 0.136654
t2_dir_thr_120     opening  test      11700       0.382021       0.333333      0.048688  0.364906  0.207978 0.156928
t2_dir_thr_120     open

Para ver solo test ordenado por mejor régimen

In [68]:
results_t2_by_regime_test = (
    results_t2_by_regime[results_t2_by_regime["split"] == "test"]
    .sort_values(["target", "bal_acc_gain", "f1_gain"], ascending=[True, False, False])
)

results_t2_by_regime_test

,target,regime_id,regime_name,split,n_samples,n_classes_train,majority_class_train,acc_model,acc_naive,acc_gain,bal_acc_model,bal_acc_naive,bal_acc_gain,f1_model,f1_naive,f1_gain,train_class_counts
9,t2_dir_thr_120,3,regular,test,41145,3,0,0.583303,0.604278,-0.020975,0.425062,0.333333,0.091728,0.424452,0.251111,0.173341,"{-1: 26907, 0: 139348, 1: 24911}"
0,t2_dir_thr_120,0,overnight,test,35100,3,0,0.640912,0.695641,-0.054729,0.387625,0.333333,0.054291,0.385112,0.273502,0.111610,"{-1: 16834, 0: 131588, 1: 14658}"
6,t2_dir_thr_120,2,opening,test,11700,3,0,0.448291,0.453419,-0.005128,0.382021,0.333333,0.048688,0.364906,0.207978,0.156928,"{-1: 10756, 0: 32133, 1: 11471}"
3,t2_dir_thr_120,1,premarket,test,11700,3,0,0.366410,0.347009,0.019402,0.357461,0.333333,0.024127,0.309111,0.171743,0.137368,"{-1: 12266, 0: 29073, 1: 13021}"
21,t2_dir_thr_90,3,regular,test,41145,3,0,0.559825,0.570349,-0.010524,0.419753,0.333333,0.086420,0.419071,0.242133,0.176938,"{-1: 27416, 0: 137503, 1: 26247}"
12,t2_dir_thr_90,0,overnight,test,35100,3,0,0.673476,0.735328,-0.061852,0.394515,0.333333,0.061182,0.395301,0.282493,0.112808,"{-1: 13810, 0: 137146, 1: 12124}"
18,t2_dir_thr_90,2,opening,test,11700,3,0,0.436496,0.435214,0.001282,0.381875,0.333333,0.048541,0.363394,0.202160,0.161234,"{-1: 12094, 0: 29637, 1: 12629}"
15,t2_dir_thr_90,1,premarket,test,11700,3,0,0.369402,0.359573,0.009829,0.354105,0.333333,0.020772,0.313171,0.176317,0.136855,"{-1: 12478, 0: 28186, 1: 13696}"


## **5.5. Conclusión — Análisis por régimen (targets T2)**


El análisis segmentado por régimen de mercado confirma que la señal predictiva identificada en los targets T2 no es homogénea, sino que depende significativamente del contexto intradía.

En primer lugar, el régimen **regular** (10:30–15:30) se destaca como el más consistente y con mayor señal explotable en ambos horizontes (`90` y `120`). En este régimen se observan las mayores mejoras en `balanced_accuracy` (~+0.09) y `f1_macro` (~+0.17), tanto en train como en test, lo que indica una estructura predictiva clara y estable. Este resultado es particularmente relevante dado que además concentra la mayor cantidad de datos, lo que refuerza su robustez.

El régimen **overnight** presenta una señal moderada, con mejoras consistentes pero menores (~+0.05 a +0.06 en balanced accuracy). Aunque el modelo logra capturar cierta estructura, la calidad de la señal es inferior a la observada en el régimen regular.

El régimen **opening** muestra una señal intermedia. Si bien las mejoras en `balanced_accuracy` son moderadas (~+0.04 a +0.05), el incremento en `f1_macro` es relativamente alto (~+0.15–0.16), lo que sugiere que el modelo logra diferenciar mejor las clases, aunque con menor estabilidad que en el régimen regular.

Por último, el régimen **premarket** presenta la señal más débil. Las mejoras en `balanced_accuracy` son marginales (~+0.02), aunque el `f1_macro` muestra incrementos relevantes. Esto sugiere que, si bien hay cierta capacidad de clasificación, la señal es menos robusta y potencialmente más ruidosa.

Un aspecto importante es que estos patrones se repiten de forma consistente en ambos horizontes (`90` y `120`), lo que refuerza la validez del resultado y sugiere que la dependencia por régimen es una propiedad estructural del problema, no un artefacto puntual.

En conjunto, estos resultados indican que:

* la señal del target T2 es **dependiente del régimen de mercado**,
* el régimen **regular concentra la mayor parte del edge**,
* los regímenes **opening y overnight aportan señal secundaria**,
* el régimen **premarket presenta la menor calidad predictiva**.

---

**Implicación para el modelado**

Este resultado justifica considerar estrategias diferenciadas por régimen, tales como:

* entrenar modelos específicos por régimen,
* ponderar observaciones según régimen,
* o incluso excluir regímenes de baja señal (como premarket) en etapas posteriores.

Como siguiente paso lógico, conviene validar si modelos no lineales (por ejemplo, boosting) amplifican estas diferencias y si el edge en el régimen regular se traduce en mejoras económicas en backtesting.


# **8. Guardado de datasets**


## **8.1. Definición de rutas**

In [86]:
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/04_features/mnq_t2.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "data/04_features/mnq_t2_summary.json"))
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## **8.2. Guardado de dataset**

In [87]:
for path, df in [
    (OUT_PARQUET, mnq_t2),
    ]:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=True)
    print(f"[OK] Guardado: {path}")

[OK] Guardado: /content/drive/MyDrive/neural_profit/data/04_features/mnq_t2.parquet


## **8.3. Generación y guardado de summary**

In [88]:
import json
import numpy as np
import pandas as pd
from pathlib import Path


def build_mnq_t2_summary(
    df: pd.DataFrame,
    *,
    out_json_path: str | Path,
    aux_cols: list[str],
    feature_cols: list[str],
    target_cols: list[str],
    date_col: str = "date",
    regime_col: str = "regime_id",
    verbose: bool = True,
) -> dict:
    """
    Genera un summary completo del dataset mnq_t2 y lo guarda en JSON.

    Incluye:
    - shape y columnas
    - rango temporal
    - número de sesiones
    - NaNs por grupo de columnas
    - distribución de targets
    - distribución de regímenes
    - estadísticas básicas de variables numéricas
    """

    df = df.copy()

    # ------------------------------------------------------------------
    # Validaciones
    # ------------------------------------------------------------------
    required_cols = aux_cols + feature_cols + target_cols
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas en el dataset: {missing}")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser DatetimeIndex")

    if not df.index.is_monotonic_increasing:
        raise ValueError("El índice no está ordenado cronológicamente")

    # ------------------------------------------------------------------
    # Info general
    # ------------------------------------------------------------------
    summary = {}

    summary["shape"] = {
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
    }

    summary["time_range"] = {
        "start": str(df.index.min()),
        "end": str(df.index.max()),
    }

    summary["sessions"] = {
        "n_sessions": int(df[date_col].nunique()),
    }

    # ------------------------------------------------------------------
    # Column groups
    # ------------------------------------------------------------------
    summary["column_groups"] = {
        "auxiliary": aux_cols,
        "features": feature_cols,
        "targets": target_cols,
    }

    # ------------------------------------------------------------------
    # NaNs
    # ------------------------------------------------------------------
    def _nan_report(cols):
        return {
            "n_nan_total": int(df[cols].isna().sum().sum()),
            "n_nan_by_col": {
                col: int(df[col].isna().sum()) for col in cols
            }
        }

    summary["nan_report"] = {
        "auxiliary": _nan_report(aux_cols),
        "features": _nan_report(feature_cols),
        "targets": _nan_report(target_cols),
    }

    # ------------------------------------------------------------------
    # Distribución de targets
    # ------------------------------------------------------------------
    target_distribution = {}

    for tgt in target_cols:
        vc = df[tgt].value_counts(dropna=False).sort_index()
        total = vc.sum()

        target_distribution[tgt] = {
            str(int(k)) if pd.notna(k) else "nan": {
                "count": int(v),
                "pct": float(v / total) if total > 0 else np.nan,
            }
            for k, v in vc.items()
        }

    summary["target_distribution"] = target_distribution

    # ------------------------------------------------------------------
    # Distribución de régimen
    # ------------------------------------------------------------------
    regime_name_map = {
        0: "overnight",
        1: "premarket",
        2: "opening",
        3: "regular",
        4: "closing",
    }

    vc_regime = df[regime_col].value_counts().sort_index()
    total_regime = vc_regime.sum()

    summary["regime_distribution"] = {
        int(k): {
            "name": regime_name_map.get(int(k), "unknown"),
            "count": int(v),
            "pct": float(v / total_regime),
        }
        for k, v in vc_regime.items()
    }

    # ------------------------------------------------------------------
    # Estadísticas numéricas
    # ------------------------------------------------------------------
    numeric_cols = feature_cols + [
        c for c in aux_cols if "delta" in c
    ]

    stats = {}

    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) == 0:
            continue

        desc = series.describe()

        stats[col] = {
            "mean": float(desc["mean"]),
            "std": float(desc["std"]),
            "min": float(desc["min"]),
            "p25": float(desc["25%"]),
            "p50": float(desc["50%"]),
            "p75": float(desc["75%"]),
            "max": float(desc["max"]),
        }

    summary["numeric_stats"] = stats

    # ------------------------------------------------------------------
    # Guardar JSON
    # ------------------------------------------------------------------
    out_json_path = Path(out_json_path)
    out_json_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_json_path, "w") as f:
        json.dump(summary, f, indent=4)

    if verbose:
        print("=" * 100)
        print("SUMMARY MNQ_T2 GENERADO")
        print("=" * 100)
        print(f"Filas                : {summary['shape']['n_rows']:,}")
        print(f"Columnas             : {summary['shape']['n_cols']:,}")
        print(f"Sesiones             : {summary['sessions']['n_sessions']:,}")
        print(f"Desde                : {summary['time_range']['start']}")
        print(f"Hasta                : {summary['time_range']['end']}")
        print(f"Archivo              : {out_json_path}")
        print("=" * 100)

    return summary

In [89]:
aux_cols = [
    'date', 'open', 'high', 'low', 'close', 'volume',
    'minute_of_day', 'close_fwd_90', 'delta_90',
    'close_fwd_120', 'delta_120'
]

feature_cols = [
    'regime_id', 'roc_30', 'roc_60',
    'stoch_k_30', 'atr_norm_10'
]

target_cols = [
    't2_dir_thr_90', 't2_dir_thr_120'
]

mnq_t2_summary = build_mnq_t2_summary(
    mnq_t2,
    out_json_path=OUT_SUMMARY,
    aux_cols=aux_cols,
    feature_cols=feature_cols,
    target_cols=target_cols,
)

SUMMARY MNQ_T2 GENERADO
Filas                : 661,745
Columnas             : 18
Sesiones             : 1,295
Desde                : 2020-01-02 05:30:00-05:00
Hasta                : 2025-06-13 14:00:00-04:00
Archivo              : /content/drive/MyDrive/neural_profit/data/04_features/mnq_t2_summary.json
